In [8]:
!pip install --upgrade transformers peft trl liger-kernel --upgrade torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.9 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 2.19.2
    Uninstalling datasets-2.19.2:
      Successfully uninstalled datasets-2.19.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.


In [2]:
!pip install "datasets<2.20.0"

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.19
    Uninstalling multiprocess-0.70.19:
      Successfully uninstalled multiprocess-0.70.19
  Attempting uninstall: datasets
    F

In [14]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

print("Memulai proses persiapan fine-tuning...")

#Load Dataset (Menggunakan cache lokal yang sudah berhasil diunduh)
dataset = load_dataset("IndoNLP/NusaX-senti", "jav", trust_remote_code=True)

def format_row(example):
    example["text"] = f"Teks Jawa: {example['text']}\nLabel Sentimen: {example['label']}"
    return example

formatted_train_dataset = dataset["train"].map(format_row)

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)

#Konfigurasi LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

training_args = SFTConfig(
    output_dir="./hasil_model_jawa",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    optim="adamw_torch",
    logging_steps=1,
    learning_rate=2e-4,
    max_steps=10,
    max_length=128,
    fp16=True,
    bf16=False,
)

print("Mempersiapkan SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_train_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print("Memulai training...")
trainer.train()

trainer.model.save_pretrained("./hasil_model_jawa")
print("Selesai! Model disimpan di folder './hasil_model_jawa'.")

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'IndoNLP/NusaX-senti' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'IndoNLP/NusaX-senti' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Memulai proses persiapan fine-tuning...


Using the latest cached version of the dataset since IndoNLP/NusaX-senti couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'jav' at /root/.cache/huggingface/datasets/IndoNLP___nusa_x-senti/jav/1.0.0/3477a395c5c7a09a74d897ceb96ebd2c3afbd1d7fad0c11d8b8026b8b08e3af5 (last modified on Thu Oct  8 14:00:02 2026).


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

...

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Memulai training...


Step,Training Loss
1,6.488936
2,6.932090
3,6.424751
4,6.077844
5,6.402124
6,6.935831
7,6.828049
8,7.168504
9,6.757314
10,6.120809


Selesai! Model disimpan di folder './hasil_model_jawa'.
